> 在线讲义：[第 50 章 · 计算与状态的分布方式：八张卡怎么接每秒 3.5 个请求](https://wikieden.github.io/ai-infra-from-constraints/lessons/50-distribution.html) · 本书《从约束推导 AI Infra》

# 第 50 章 · 计算与状态的分布方式：八张卡怎么接每秒 3.5 个请求

配套讲义：仓库里 `ai-infra/notes/lessons/50-distribution.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。



四张 A100 + 四张 H20、每秒 3.5 个 8K 输入 / 1025 输出请求、Qwen3-8B BF16，取自 AII 第 9 章 §9.1。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
GiB = 2**30
K_TOKEN = 2 * 36 * 8 * 128 * 2  # Qwen3-8B 每 token KV 147,456 B = 144 KiB
K_235B = 2 * 94 * 4 * 128 * 2   # Qwen3-235B-A22B 每 token KV 192,512 B = 188 KiB
D_P = {"A100": 0.8564, "H20": 1.8054}
D_D = {"A100": 1.7634, "H20": 0.8782}

**函数 `gpus_needed`**：λ d：持续工作所需的资源单元数，需要 < n_r。

In [ ]:
def gpus_needed(lam: float, d: float) -> float:
    """λ d：持续工作所需的资源单元数，需要 < n_r。"""
    return lam * d

**函数 `colocated_rate`**：共置副本的请求率上限：Σ N / (d_P + d_D)。

In [ ]:
def colocated_rate(cards: dict[str, int]) -> float:
    """共置副本的请求率上限：Σ N / (d_P + d_D)。"""
    return sum(n / (D_P[c] + D_D[c]) for c, n in cards.items())

**函数 `kv_gib`**：小工具。

In [ ]:
def kv_gib(tokens: int, copies: int = 1, per_token: int = K_TOKEN) -> float:
    return tokens * per_token * copies / GiB

**函数 `residency`**：容量 × 时间（GiB·s）。

In [ ]:
def residency(gib: float, seconds: float) -> float:
    """容量 × 时间（GiB·s）。"""
    return gib * seconds

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：资源约束 λ d < n")
print(f"  每秒 4 个请求、每个 0.5 GPU 秒 → 需要 {gpus_needed(4, 0.5):.1f} 张持续工作的卡")
for c in D_P:
    print(f"  {c}：P {D_P[c]:.3f} + D {D_D[c]:.3f} = {D_P[c] + D_D[c]:.2f} GPU 秒/请求")
mu = colocated_rate({"A100": 4, "H20": 4})
print(f"  八个完整副本：{mu:.2f} 请求/s，到达 3.5 → 每秒积压 {3.5 - mu:.2f}")

In [ ]:
print("\n第二步：TP2×EP4 的一个实例")
print(f"  Qwen3-235B 注意力与 KV 在四个 EP 组各复制一份：8K 上下文 {kv_gib(8192, 1, K_235B):.3f} GiB × 4 = {kv_gib(8192, 4, K_235B):.3f} GiB")

In [ ]:
print("\n第三步：状态驻留")
one = kv_gib(8192)
print(f"  8K 上下文 {one:.3f} GiB，工具等 10 s → {residency(one, 10):.2f} GiB·s；十个会话同时等 → {10 * one:.2f} GiB")
print(f"  返回 1025 个 token 后 KV 覆盖 {8192 + 1024} 个 token（最后一个还没送回模型）")

## 3. 代码里才看得到的

1. `D_P`、`D_D` 用的是第 51 章推出的未取整值（0.8564、1.7634 …），所以积压是 0.48 而不是正好 0.5。
2. Qwen3-235B 每 token KV 是 $2 \times 94 \times 4 \times 128 \times 2 = 192{,}512$ B，和 Qwen3-8B 的 144 KiB 不同，别混用。
3. `gpus_needed` 只给平均需要几张；有突发时，刚好等于就会一直积压。

## 4. 自己改一改

- 给 `colocated_rate` 传 `{"H20": 8}`，和 4 + 4 比。
- 把 `D_D` 里 H20 的值调大 20%，看八副本的缺口变成多少。
- 写一个函数：给定到达率，返回最少要几张 A100 + 几张 H20 才能共置接住。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 洗车每辆占半小时，来得快就要多工位；
> 全能队、分组队、轮流进屋、共用资料室，各改一处；
> 客人接电话，桌子照样留。